# Stage 2: data, trainers and pipeline checks

Each section restarts on its own and saves to Drive. Before each session: push from your laptop, then run section 0, which refuses to continue from code that isn't pushed.

| Section | Runtime | Colab units (estimate) |
|---|---|---|
| 0. Setup | Same as the section you're running | About 0 |
| A. Data and GPT-4o solutions | **CPU, no GPU needed** | About 0 |

Sections B to F (mini-pilot, memory test, smoke runs, lengths and drift, decision inputs) come with the trainer batch. **Commit the `prereg-draft` tag before running section B.**

## 0. Setup

In [ ]:
import os

if not os.path.isdir("/content/kl-reseach"):
    !git clone https://github.com/JuaniLlaberia/kl-reseach.git /content/kl-reseach
%cd /content/kl-reseach
!git pull --ff-only
!pip install -q -r requirements.txt
!pip uninstall -q -y torchaudio torchvision

Checks the pinned versions and the commit. If it says torch is on the wrong version, restart the session and rerun this cell.

In [ ]:
%cd /content/kl-reseach
import subprocess
from pathlib import Path

import torch

from utils.env import check_pins, check_pushed, read_pins

REPO = Path("/content/kl-reseach")

installed = check_pins(REPO / "requirements.txt")
torch_pin = read_pins(REPO / "requirements.txt")["torch"]
if torch.__version__.split("+")[0] != torch_pin:
    raise RuntimeError(f"torch {torch.__version__} is loaded. Restart the session.")

COMMIT = check_pushed(REPO)
print("Commit:", COMMIT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print(installed)

Files in `/content` are lost when the runtime is recycled, so everything is saved to Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/kl-reseach")

Runs the test suite (CPU, no network). Nothing below should run if this fails.

In [ ]:
tests = subprocess.run(["python", "-m", "pytest", "tests/", "-q", "-p", "no:cacheprovider"], capture_output=True, text=True, cwd=REPO)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("The tests failed. Don't run anything until they pass.")

## A. Data and GPT-4o solutions

Use a **CPU runtime** for this section, as nothing here needs a GPU.

Add your OpenRouter key under Colab secrets as `OPENROUTER_API_KEY` and enable it for this notebook.

### Chemistry L-3 rows

Checks the row counts against the lab log, and that the split gives exactly the committed file.

In [ ]:
from collections import Counter

from src.data.chem import check_counts, load_chem_rows, load_split, select_rows, split_rows

SPLIT_PATH = REPO / "data" / "splits" / "chem_l3.json"

rows = load_chem_rows()
check_counts(rows)
split = load_split(SPLIT_PATH)
if split_rows(rows) != split:
    raise RuntimeError("The split doesn't reproduce the committed file.")

train_rows = select_rows(rows, split, "train")
heldout_rows = select_rows(rows, split, "heldout")
kl_rows = select_rows(rows, split, "kl")

print("Rows per type:", dict(Counter(row["type"] for row in rows)))
print("Rows per task:", dict(Counter(row["details"]["task"] for row in rows)))
print("Train / held-out / KL:", len(train_rows), len(heldout_rows), len(kl_rows))

### 50-prompt test

One first try each on 50 train rows. It costs about $0.15 by my estimate, and it counts towards the $15 limit. Its attempts stay in the log, so the full run continues from them instead of paying twice.

**Stop after this and check the output with Claude before the full run.**

In [ ]:
import json

from google.colab import userdata

from src.data.chem import build_prompt, stratified_sample
from src.data.solutions import generate_solutions, load_attempts, project_full_run, summarize_attempts

API_KEY = userdata.get("OPENROUTER_API_KEY")
BUDGET_USD = 15.0
TEST_MAX_TOKENS = 1024
SOLUTIONS_DIR = DRIVE_ROOT / "data" / "solutions"
LOG_PATH = SOLUTIONS_DIR / "chem_l3_gpt4o_attempts.jsonl"

test_rows = stratified_sample(train_rows, 50, seed=1)
result = generate_solutions(test_rows, split, LOG_PATH, API_KEY, BUDGET_USD, TEST_MAX_TOKENS, max_tries=1)

test_ids = {row["row_id"] for row in test_rows}
test_attempts = [record for record in load_attempts(LOG_PATH) if record["row_id"] in test_ids and record["attempt"] == 1]
summary = summarize_attempts(test_rows, test_attempts)
projection = project_full_run(train_rows, test_rows, test_attempts, BUDGET_USD)

print(result)
print(json.dumps(summary, indent=2))
print(json.dumps(projection, indent=2))

Read a few responses, to check that the wording works and that nothing valid is being rejected. Look at the wrong ones and those with no valid answer.

In [ ]:
from src.data.chem import get_gold
from src.eval.extract import extract_answer

by_id = {row["row_id"]: row for row in test_rows}
shown = Counter()
for record in test_attempts:
    row = by_id[record["row_id"]]
    answer = extract_answer(record["text"], row["type"])
    kind = "valid answer, correct" if record["correct"] else ("no valid answer" if answer is None else "valid answer, wrong")
    shown[kind] += 1
    if shown[kind] <= 3 or row["type"] == "filling":
        print("=" * 80)
        print(f"{kind} | {row['details']['task']} | gold: {get_gold(row)} | extracted: {answer} | {record['completion_tokens']} tokens")
        print(build_prompt(row))
        print("-" * 40)
        print(record["text"][-600:])
print(shown)

### Full run

**Only run this after checking the test together.** Set `APPROVED_FULL_RUN` to `True`, and set `FULL_MAX_TOKENS` from the test's token counts.

If it stops or crashes, rerun the cell: it continues from the log without repeating any paid call. The limit covers everything in the log.

If the prompt wording changes after the test, rename the log and pass what it cost as `prior_spend_usd`, as the code refuses to mix wordings.

In [ ]:
APPROVED_FULL_RUN = False
FULL_MAX_TOKENS = None

if not APPROVED_FULL_RUN or FULL_MAX_TOKENS is None:
    raise RuntimeError("Not approved yet.")

result = generate_solutions(train_rows, split, LOG_PATH, API_KEY, BUDGET_USD, FULL_MAX_TOKENS)
print(result)

### Solutions and log entry

Saves one solution per solved row, plus the summary for the lab log. Afterwards, copy `chem_l3_gpt4o.jsonl`, the attempts log and the summary from Drive to `data/solutions/` in the repo and commit them.

Compare the total against your OpenRouter dashboard.

In [ ]:
from src.data.solutions import save_solutions

attempts = load_attempts(LOG_PATH)
n_solutions = save_solutions(train_rows, attempts, SOLUTIONS_DIR / "chem_l3_gpt4o.jsonl")
summary = summarize_attempts(train_rows, attempts)
summary["commit"] = COMMIT
(SOLUTIONS_DIR / "chem_l3_gpt4o_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Solved {n_solutions} of {len(train_rows)} ({n_solutions / len(train_rows):.1%})")
print(f"Total: ${summary['cost_usd']:.2f} in {summary['calls']} calls")
print(json.dumps(summary, indent=2))